In [13]:
from bs4 import BeautifulSoup
import warnings
warnings.filterwarnings('ignore')
import requests
import json
import pandas as pd
import time

In [ ]:
# function for scraping one firstcry category
# 50 pages = around 1000 products

def scrape_firstcry_category(search_string, total_pages=50):

    # headers for api request
    headers = {
        'sec-ch-ua-platform': '"macOS"',
        'Referer': 'https://www.firstcry.com/diapering/1/0/0?sort=bestseller&ref2=menu_dd_catlanding',
        'sec-ch-ua': '"Chromium";v="152", "Not?A_Brand";v="24", "Google Chrome";v="152"',
        'sec-ch-ua-mobile': '?0',
        'X-Requested-With': 'XMLHttpRequest',
        'User-Agent': 'Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/152.0.0.0 Safari/537.36',
        'Accept': 'application/json, text/javascript, */*; q=0.01',
        'Content-Type': 'application/json; charset=utf-8'
    }

    # empty list for all products
    all_products = []

    # going through pages one by one
    for page in range(1, total_pages + 1):

        # api url for each page
        url = (
            f'https://www.firstcry.com/svcs/SearchResult.svc/GetSearchResultProductsPaging'
            f'?PageNo={page}&PageSize=20&SortExpression=bestseller&OnSale=1'
            f'&SearchString={search_string}'
            f'&SubCatId=&BrandId=&Price=&Age=&Color=&OptionalFilter=&OutOfStock=&Type1='
            f'&Type2=&Type3=&Type4=&Type5=&Type6=&Type7=&Type8=&Type9=&Type10='
            f'&Type11=&Type12=&Type13=&Type14=&Type15=&combo=&discount='
            f'&searchwithincat=&ProductidQstr=&searchrank=&pmonths=&cgen='
            f'&PriceQstr=&DiscountQstr=&sorting=&MasterBrand=0&Rating=&Offer='
            f'&skills=&material=&curatedcollections=&measurement=&gender='
            f'&exclude=&premium=&pcode=0&isclub=0&deliverytype=&authors='
            f'&booktype=&character=&collections=&format=&genre='
            f'&booklanguage=&publication=&skill='
        )

        try:
            # sending request
            response = requests.get(url, headers=headers, timeout=10)

            # checking request worked
            if response.status_code != 200:
                print(f"Page {page} failed with status {response.status_code}. Stopping.")
                break

            # getting json response
            outer = response.json()

            # json inside json
            inner = json.loads(outer['ProductResponse'])

            # getting products list
            products = inner.get('Products', [])

            # stop if no products
            if not products:
                print(f"No products found on page {page}. Category complete.")
                break

            # taking required details from each product
            for p in products:
                all_products.append({
                    'product_name':   p.get('PNm'),
                    'brand':          p.get('BNm'),
                    'category':       p.get('CNm'),
                    'subcategory':    p.get('SCNm'),
                    'size':           p.get('size'),
                    'mrp':            float(p.get('MRP', 0)),
                    'discount_pct':   float(p.get('Disc', 0)),
                    'selling_price':  float(p.get('discprice', 0)),
                    'price_per_unit': float(p.get('PUCost', 0)),
                    'stock':          int(p.get('CrntStock', 0)),
                    'rating':         float(p.get('rating', 0)),
                    'review_count':   int(p.get('review', 0)),
                    'source':         'FirstCry'
                })

            # checking progress
            print(f"Page {page} done — {len(products)} products collected.")

            # small gap between requests
            time.sleep(1)

        except Exception as e:
            print(f"Error on page {page}: {e}")
            break

    # converting product list into dataframe
    return pd.DataFrame(all_products)

In [14]:
# categories we want to scrape

categories = {
    'diapering': 'diapering',
    'baby_food':  'baby food',
    'baby_care':  'baby care'
}

# empty list for category dataframes
all_dfs = []

# scraping each category
for name, search_term in categories.items():

    print(f"\nScraping category: {name}")

    # calling function
    df = scrape_firstcry_category(search_term, total_pages=50)

    # adding category used for search
    df['search_category'] = name

    # storing dataframe
    all_dfs.append(df)

    print(f"Total products collected for {name}: {len(df)}")


# combining all categories
final_df = pd.concat(all_dfs, ignore_index=True)

# removing duplicate products
final_df.drop_duplicates(
    subset='product_name',
    inplace=True
)

# saving final data
final_df.to_csv(
    'firstcry_competitor_data.csv',
    index=False
)

print(f"\nDone. Total unique products saved: {len(final_df)}")


Scraping category: diapering
Page 1 done — 20 products collected.
Page 2 done — 20 products collected.
Page 3 done — 20 products collected.
Page 4 done — 20 products collected.
Page 5 done — 20 products collected.
Page 6 done — 20 products collected.
Page 7 done — 20 products collected.
Page 8 done — 20 products collected.
Page 9 done — 20 products collected.
Page 10 done — 20 products collected.
Page 11 done — 20 products collected.
Page 12 done — 20 products collected.
Page 13 done — 20 products collected.
Page 14 done — 20 products collected.
Page 15 done — 20 products collected.
Page 16 done — 20 products collected.
Page 17 done — 20 products collected.
Page 18 done — 20 products collected.
Page 19 done — 20 products collected.
Page 20 done — 20 products collected.
Page 21 done — 20 products collected.
Page 22 done — 20 products collected.
Page 23 done — 20 products collected.
Page 24 done — 20 products collected.
Page 25 done — 20 products collected.
Page 26 done — 20 products co